# Task 3 — live generation launcher (Step 4)

Thin launcher only: logic in `task3/live_generate.py`. Settings: **GPU T4 x2**, **Internet on**, secret **HF_TOKEN**. The 50 questions are split over both GPUs; finished questions are skipped on rerun (results synced to the HF branch `task3-live` after every question).

In [ ]:
REPO_URL= "https://github.com/Zeeba1705/atml_PA3.git"
BRANCH= "task3-ttc"
SYNC_REPO= "zeeba1705/atml-pa3"

In [ ]:
import os
import subprocess
import sys
import time

if not os.path.exists("/kaggle/working/atml_PA3"):
    !git clone -b {BRANCH} {REPO_URL} /kaggle/working/atml_PA3

%cd /kaggle/working/atml_PA3
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
#stop here without both GPUs (Settings -> Accelerator -> GPU T4 x2)
assert torch.cuda.device_count() == 2, f"need GPU T4 x2, found {torch.cuda.device_count()} GPUs"

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"]= UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PYTHONUNBUFFERED"]= "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]= "1"

### Two shards, one per T4
Logs in `logs/live{0,1}.log`, copied here every 2 min.

In [ ]:
os.makedirs("logs", exist_ok=True)

procs= {}
for shard in [0, 1]:
    env= dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
    log= open(f"logs/live{shard}.log", "a")
    procs[shard]= subprocess.Popen(
        [
            sys.executable, "-m", "task3.live_generate",
            "--shard", str(shard),
            "--n_shards", "2",
            "--sync_repo", SYNC_REPO
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env
    )

seen= {s: 0 for s in procs}
while True:
    running= any(p.poll() is None for p in procs.values())
    time.sleep(120 if running else 0)

    for s in procs:
        with open(f"logs/live{s}.log") as f:
            f.seek(seen[s])
            new= f.read()
            seen[s]= f.tell()
        for line in new.splitlines():
            print(f"[gpu{s}] {line}")

    if not running:
        break

print({s: p.returncode for s, p in procs.items()})